In [6]:
import sys
sys.path.append("/Users/katherinegeng/Desktop/projects/pulsar")

In [1]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from bertopic import BERTopic
import numpy as np

In [2]:
def evaluate_topic_model(topic_model, abstracts):
    topics = topic_model.get_topics()
    
    topic_words = []
    for topic_id, words in topics.items():
        if topic_id == -1:
            continue
        top_words = [word for word, _ in words[:10]]
        topic_words.append(top_words)
    
    tokenized = [abstract.lower().split() for abstract in abstracts]
    dictionary = Dictionary(tokenized)
    corpus = [dictionary.doc2bow(doc) for doc in tokenized]
    
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized,
        dictionary=dictionary,
        coherence="c_v"
    )
    cv_score = coherence_model.get_coherence()
    
    all_words = []
    for words in topic_words:
        all_words.extend(words)
    
    diversity = len(set(all_words)) / len(all_words) if all_words else 0
    
    return {
        "num_topics": len(topic_words),
        "cv_coherence": round(cv_score, 4),
        "topic_diversity": round(diversity, 4),
        "noise_ratio": list(topic_model.get_document_info(abstracts)["Topic"]).count(-1) / len(abstracts)
    }

In [7]:
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from src.features.scores import filtered_papers

In [9]:
papers = filtered_papers()
abstracts = []

for paper in papers:
    abstracts.append(paper["abstract"])

In [11]:
configurations = [
    {"min_cluster_size": 5,  "min_samples": 2, "n_neighbors": 15},
    {"min_cluster_size": 8,  "min_samples": 3, "n_neighbors": 15},
    {"min_cluster_size": 10, "min_samples": 3, "n_neighbors": 15},
    {"min_cluster_size": 5,  "min_samples": 2, "n_neighbors": 10},
    {"min_cluster_size": 5,  "min_samples": 2, "n_neighbors": 20},
]

results = []

for config in configurations:
    umap_model = UMAP(
        n_neighbors=config["n_neighbors"],
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=config["min_cluster_size"],
        min_samples=config["min_samples"],
        metric="euclidean",
        cluster_selection_method="eom"
    )
    vectorizer = CountVectorizer(stop_words="english")
    
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer,
        calculate_probabilities=False
    )
    
    topics, _ = topic_model.fit_transform(abstracts)
    metrics = evaluate_topic_model(topic_model, abstracts)
    metrics["config"] = config
    results.append(metrics)
    
    print(f"Config: {config}")
    print(f"Topics: {metrics['num_topics']}, Cv: {metrics['cv_coherence']}, Diversity: {metrics['topic_diversity']}, Noise: {metrics['noise_ratio']:.2%}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Config: {'min_cluster_size': 5, 'min_samples': 2, 'n_neighbors': 15}
  Topics: 23, Cv: 0.3387, Diversity: 0.9174, Noise: 17.55%



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Config: {'min_cluster_size': 8, 'min_samples': 3, 'n_neighbors': 15}
  Topics: 17, Cv: 0.3461, Diversity: 0.9353, Noise: 19.95%



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Config: {'min_cluster_size': 10, 'min_samples': 3, 'n_neighbors': 15}
  Topics: 13, Cv: 0.3486, Diversity: 0.9231, Noise: 17.55%



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Config: {'min_cluster_size': 5, 'min_samples': 2, 'n_neighbors': 10}
  Topics: 26, Cv: 0.3349, Diversity: 0.9462, Noise: 15.16%



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Config: {'min_cluster_size': 5, 'min_samples': 2, 'n_neighbors': 20}
  Topics: 31, Cv: 0.3291, Diversity: 0.9387, Noise: 25.00%

